<a href="https://colab.research.google.com/github/ElenaCoutoFraga/PIIA_AgenteConversacional/blob/main/pruebawebsocket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**VAD**

In [ ]:
!pip install -q torchaudio
!pip install faster-whisper
!pip install groq
! pip install -U pip
! pip install coqui-tts
! pip install python-dotenv
! pip install fastapi uvicorn websockets

import torch
torch.set_num_threads(1)
from IPython.display import Audio
from pprint import pprint
from faster_whisper import WhisperModel
import matplotlib.pyplot as plt
from TTS.api import TTS
import os
from groq import Groq
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# --------------- DEFINICIÓN DEL VAD ---------------
USE_PIP = True # download model using pip package or torch.hub
USE_ONNX = False # change this to True if you want to test onnx model
if USE_ONNX:
    !pip install -q onnxruntime
if USE_PIP:
  !pip install -q silero-vad
  from silero_vad import (load_silero_vad,
                          read_audio,
                          get_speech_timestamps,
                          save_audio,
                          VADIterator,
                          collect_chunks)
  VAD = load_silero_vad(onnx=USE_ONNX)
else:
  VAD, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                model='silero_vad',
                                force_reload=True,
                                onnx=USE_ONNX)

  (get_speech_timestamps,
  save_audio,
  read_audio,
  VADIterator,
  collect_chunks) = utils


In [ ]:
# --------------- DEFINICIÓN DEL ASR ---------------
# Especificamos el modelo
model_size = "base"

# Run on GPU with FP16
ASR = WhisperModel(model_size, device="cuda", compute_type="float16")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# --------------- DEFINICIÓN DEL LLM ---------------
load_dotenv("/content/.env", override=True)

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("No se encontró GROQ_API_KEY en /content/.env")

client = Groq(api_key=api_key.strip())

In [ ]:
# --------------- DEFINICIÓN DEL TTS ---------------
ts = TTS("tts_models/es/css10/vits", gpu=True)

/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


In [ ]:
def run_vad(ruta_audio, SAMPLING_RATE = 16000):
  wav = read_audio(ruta_audio, sampling_rate=SAMPLING_RATE)
  speech_probs = []

  window_size_samples = 512 if SAMPLING_RATE == 16000 else 256
  vad_iterator = VADIterator(VAD, sampling_rate=SAMPLING_RATE)

  speech_events = []
  for i in range(0, len(wav), window_size_samples):
      chunk = wav[i:i + window_size_samples]
      if len(chunk) < window_size_samples:
          break

      speech_prob = VAD(chunk, SAMPLING_RATE).item()
      speech_probs.append(speech_prob)

      speech_dict = vad_iterator(chunk, return_seconds=True)
      if speech_dict:
          speech_events.append(speech_dict)

  vad_iterator.reset_states()

  return {
      "speech_probs": speech_probs,
      "speech_events": speech_events,
  }

In [ ]:
def run_asr(ruta_audio: str):
    segments, info = ASR.transcribe(ruta_audio, beam_size=5)
    text = " ".join(segment.text for segment in segments)
    return {
        "text": text,
        "language": getattr(info, "language", None)
    }

In [ ]:
def run_llm(text: str):
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "user", "content": text}
        ],
        model="llama-3.3-70b-versatile",
    )

    respuesta = chat_completion.choices[0].message.content
    return respuesta

In [ ]:
#if idioma == "ga": idioma = "es"
def run_tts(text: str, output_path: str = "output.wav"):
    ts.tts_to_file(
        text=text,
        file_path=output_path
    )
    return output_path

In [ ]:
def run_pipeline(ruta_audio: str):
    vad_result = run_vad(ruta_audio)
    asr_result = run_asr(ruta_audio)
    llm_text = run_llm(asr_result["text"])
    tts_path = run_tts(llm_text)

    return {
        "vad": vad_result,
        "asr": asr_result,
        "llm": llm_text,
        "tts_path": tts_path,
    }

In [ ]:
run_pipeline("miau.29.30.mp4")

{'vad': {'speech_probs': [0.0016982493689283729,
   0.05834949389100075,
   0.026905974373221397,
   0.03698375076055527,
   0.01954682171344757,
   0.014430217444896698,
   0.014388549141585827,
   0.012249719351530075,
   0.010167570784687996,
   0.00976551603525877,
   0.005810944363474846,
   0.0067288316786289215,
   0.004778947215527296,
   0.004305916838347912,
   0.006354071199893951,
   0.0057234750129282475,
   0.005845455918461084,
   0.004609226249158382,
   0.003617056179791689,
   0.0026981167029589415,
   0.0021060199942439795,
   0.002492599654942751,
   0.00409600418061018,
   0.003989863209426403,
   0.0044539389200508595,
   0.15187636017799377,
   0.2657926082611084,
   0.11013592034578323,
   0.03634628280997276,
   0.023709462955594063,
   0.013860714621841908,
   0.009211733005940914,
   0.007196311838924885,
   0.006303531117737293,
   0.009656522423028946,
   0.25187599658966064,
   0.08499550074338913,
   0.05802644416689873,
   0.022438600659370422,
   0.0106

## Servidor WebSocket – procesamiento de audio en continuo

El servidor expone el endpoint `ws://<host>:8765/ws/audio`.

**Protocolo**
| Dirección | Formato |
|-----------|--------|
| Cliente → Servidor | Bytes PCM crudo: **int16, mono, 16 kHz** (cualquier número de muestras por mensaje) |
| Servidor → Cliente | Bytes de un fichero **WAV** completo con la respuesta TTS |

El servidor aplica el VAD (Silero) en ventanas de 512 muestras. Cuando detecta el
**final** de un segmento de habla, ejecuta el pipeline completo (ASR → LLM → TTS)
en un hilo aparte para no bloquear el bucle de eventos y devuelve el audio sintetizado al cliente.

In [ ]:
# --------------- SERVIDOR WEBSOCKET (AUDIO EN CONTINUO) ---------------
import asyncio
import os
import tempfile
import threading

import numpy as np
import torchaudio
from fastapi import FastAPI, WebSocket
from fastapi.websockets import WebSocketDisconnect
import uvicorn

app = FastAPI()

SAMPLING_RATE = 16000
WINDOW_SIZE_SAMPLES = 512  # muestras por ventana VAD (16 kHz)


@app.websocket("/ws/audio")
async def websocket_audio_endpoint(websocket: WebSocket):
    """
    Endpoint WebSocket para procesamiento de audio en continuo.

    - Recibe: chunks de audio PCM (int16, mono, 16 kHz) como bytes.
    - Envía:  fichero WAV con la respuesta TTS tras cada segmento de habla detectado.
    """
    await websocket.accept()
    print("[WS] Cliente conectado")

    vad_iterator = VADIterator(VAD, sampling_rate=SAMPLING_RATE)
    speech_buffer = []
    is_speaking = False
    loop = asyncio.get_event_loop()

    try:
        while True:
            raw = await websocket.receive_bytes()
            # El cliente envía el texto "EOF" para señalar el fin del stream
            if raw == b"EOF":
                break

            # Convertir PCM int16 → tensor float32 normalizado [-1, 1]
            audio_int16 = np.frombuffer(raw, dtype=np.int16)
            audio_float32 = audio_int16.astype(np.float32) / 32768.0
            chunk_tensor = torch.from_numpy(audio_float32)

            # Iterar en ventanas de WINDOW_SIZE_SAMPLES muestras
            for offset in range(0, len(chunk_tensor), WINDOW_SIZE_SAMPLES):
                window = chunk_tensor[offset : offset + WINDOW_SIZE_SAMPLES]
                if len(window) < WINDOW_SIZE_SAMPLES:
                    break

                speech_event = vad_iterator(window, return_seconds=True)

                if speech_event:
                    if "start" in speech_event:
                        is_speaking = True
                        speech_buffer = [window]  # incluir la ventana que dispara el inicio
                    elif "end" in speech_event and is_speaking:
                        is_speaking = False
                        if speech_buffer:
                            full_audio = torch.cat(speech_buffer)
                            speech_buffer = []

                            # Procesar el segmento en un executor para no bloquear el event loop
                            response_bytes = await loop.run_in_executor(
                                None, _process_segment, full_audio
                            )
                            if response_bytes:
                                await websocket.send_bytes(response_bytes)
                elif is_speaking:
                    speech_buffer.append(window)

    except WebSocketDisconnect:
        print("[WS] Cliente desconectado")
    finally:
        vad_iterator.reset_states()


def _process_segment(audio_tensor: torch.Tensor):
    """
    Ejecuta el pipeline completo (ASR → LLM → TTS) sobre un segmento de habla
    y devuelve los bytes del WAV sintetizado, o None si no hay texto.
    """
    try:
        # Guardar segmento en fichero temporal para ASR
        tmp_in = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp_in.close()
        torchaudio.save(tmp_in.name, audio_tensor.unsqueeze(0), SAMPLING_RATE)

        asr_result = run_asr(tmp_in.name)
        os.unlink(tmp_in.name)

        transcript = asr_result.get("text", "").strip()
        if not transcript:
            print("[ASR] Segmento sin texto, se descarta")
            return None
        print(f"[ASR] {transcript}")

        llm_response = run_llm(transcript)
        print(f"[LLM] {llm_response}")

        # Guardar TTS en fichero temporal
        tmp_out = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp_out.close()
        run_tts(llm_response, tmp_out.name)

        with open(tmp_out.name, "rb") as f:
            wav_bytes = f.read()
        os.unlink(tmp_out.name)

        return wav_bytes

    except Exception as exc:
        print(f"[ERROR] Fallo al procesar segmento: {exc}")
        return None


def start_websocket_server(host: str = "0.0.0.0", port: int = 8765):
    """Lanza el servidor FastAPI/uvicorn en un hilo daemon en segundo plano."""
    uvicorn.run(app, host=host, port=port, log_level="info")


websocket_server_thread = threading.Thread(target=start_websocket_server, daemon=True)
websocket_server_thread.start()
print("✅ Servidor WebSocket arrancado en ws://0.0.0.0:8765/ws/audio")


## Escucha en continuo desde el micrófono

Esta sección sustituye el cliente de prueba por archivo por una captura directa del micrófono del navegador (Google Colab) e incorpora dos lógicas configurables:

- **Fin de turno** (`should_end_turn`): decide cuándo el usuario ha terminado de hablar y hay que procesar su mensaje.
- **Interrupción** (`should_interrupt`): decide si el usuario interrumpe al agente mientras está reproduciendo audio.

**Flujo de la conversación:**
1. El navegador captura el micrófono y envía chunks PCM (int16, mono, 16 kHz) por WebSocket.
2. El servidor aplica VAD frame a frame y acumula el audio mientras hay habla.
3. Cuando `should_end_turn()` devuelve `True` → se procesa el segmento (ASR → LLM → TTS).
4. El servidor envía el WAV de respuesta; el cliente lo reproduce en el navegador.
5. Si durante la reproducción el usuario habla y `should_interrupt()` devuelve `True` →
   el servidor envía `"INTERRUPT"` para detener la reproducción en el cliente y empieza a escuchar de nuevo.

> **Instrucciones:**
> 1. Ejecuta las celdas en orden (lógicas → servidor → captura JS).
> 2. En la celda JS pulsa **🎤 Iniciar** y concede permisos de micrófono al navegador.
> 3. Habla; el agente responderá en audio.

In [ ]:
# --------------- LÓGICA DE FIN DE TURNO ---------------
# Configura cuándo se considera que el usuario ha terminado su turno.
# La función está conectada al bucle VAD del servidor (celda siguiente);
# solo necesitas completar el bloque TODO con tu lógica propia.

# ── Parámetros base ──────────────────────────────────────────────────────────
SILENCE_THRESHOLD_S   = 1.5   # segundos de silencio para dar el turno por terminado
MIN_SPEECH_DURATION_S = 0.5   # duración mínima del segmento para que merezca procesarse


def should_end_turn(
    silence_duration_s: float,
    speech_duration_s: float,
    transcript_so_far: str = "",
) -> bool:
    """
    Devuelve True si se debe dar por terminado el turno del usuario.

    Parámetros
    ----------
    silence_duration_s  : segundos sin voz desde el último frame con habla
    speech_duration_s   : duración total del segmento de habla acumulado
    transcript_so_far   : transcripción parcial disponible (puede estar vacía)
    """
    # ── Condiciones mínimas (no modificar) ───────────────────────────────────
    if silence_duration_s < SILENCE_THRESHOLD_S:
        return False
    if speech_duration_s < MIN_SPEECH_DURATION_S:
        return False

    # ── TODO: añade aquí tu lógica de fin de turno ───────────────────────────
    # Dispones de silence_duration_s, speech_duration_s y transcript_so_far.
    # Algunos ejemplos orientativos (descomenta o escribe el tuyo):
    #
    #   # Detectar signo de puntuación de cierre en la transcripción parcial
    #   if transcript_so_far and transcript_so_far.strip()[-1] in ".?!":
    #       return True
    #
    #   # Exigir más silencio si el segmento de habla es muy largo
    #   if speech_duration_s > 5.0 and silence_duration_s < 2.5:
    #       return False
    #
    #   # Integrar un modelo de predicción de fin de turno (Turn-Taking)
    #   # score = turntaking_model.predict(transcript_so_far)
    #   # return score > 0.8
    #
    # ─────────────────────────────────────────────────────────────────────────

    return True


In [ ]:
# --------------- LÓGICA DE INTERRUPCIÓN ---------------
# Configura cómo reacciona el sistema cuando el usuario habla mientras el
# agente está reproduciendo audio (TTS).
# Completa los dos bloques TODO con tu lógica.

import asyncio

# ── Parámetros base ──────────────────────────────────────────────────────────
INTERRUPTION_MIN_FRAMES     = 3     # frames consecutivos con voz para confirmar interrupción
INTERRUPTION_PROB_THRESHOLD = 0.7   # probabilidad VAD media mínima para considerar habla real


def should_interrupt(
    speech_frames_count: int,
    avg_speech_prob: float,
    current_tts_text: str = "",
) -> bool:
    """
    Devuelve True si se debe interrumpir la reproducción del TTS.

    Parámetros
    ----------
    speech_frames_count : frames consecutivos con voz detectados durante el TTS
    avg_speech_prob     : probabilidad VAD media en esos frames
    current_tts_text    : texto que está leyendo el agente en este momento
    """
    # ── Condiciones mínimas (no modificar) ───────────────────────────────────
    if speech_frames_count < INTERRUPTION_MIN_FRAMES:
        return False
    if avg_speech_prob < INTERRUPTION_PROB_THRESHOLD:
        return False

    # ── TODO: añade aquí tu lógica de interrupción ───────────────────────────
    # Dispones de speech_frames_count, avg_speech_prob y current_tts_text.
    # Algunos ejemplos orientativos (descomenta o escribe el tuyo):
    #
    #   # Requerir una palabra clave para interrumpir (necesita ASR parcial)
    #   # if "para" not in partial_transcript.lower():
    #   #     return False
    #
    #   # No interrumpir si el agente está en un momento clave del discurso
    #   # if any(kw in current_tts_text.lower() for kw in ["importante", "atención"]):
    #   #     return False
    #
    #   # Backoff: ignorar interrupciones muy seguidas en el tiempo
    #   # global _last_interruption_time
    #   # if time.time() - _last_interruption_time < 2.0:
    #   #     return False
    #   # _last_interruption_time = time.time()
    #
    # ─────────────────────────────────────────────────────────────────────────

    return True


async def on_interruption_triggered(websocket, loop: asyncio.AbstractEventLoop):
    """
    Acción ejecutada cuando `should_interrupt` devuelve True.
    Envía "INTERRUPT" al cliente para detener la reproducción y prepara el
    servidor para escuchar el nuevo turno del usuario.

    TODO: añade aquí acciones adicionales si lo necesitas.
    Algunos ejemplos orientativos:
    #   - Limpiar una cola de respuestas LLM pendientes
    #   - Registrar la interrupción para análisis posterior
    #   - Enviar un mensaje de texto de cortesía al usuario
    """
    # ── Acción base: notificar al cliente (no modificar) ─────────────────────
    asyncio.run_coroutine_threadsafe(
        websocket.send_text("INTERRUPT"),
        loop,
    )
    print("[WS] Interrupción detectada – reproducción detenida en el cliente")

    # ── TODO: añade aquí acciones adicionales ────────────────────────────────
    #
    # ─────────────────────────────────────────────────────────────────────────


In [ ]:
# --------------- SERVIDOR WEBSOCKET CON MICRÓFONO (LÓGICAS INTEGRADAS) ---------------
# Servidor actualizado que recibe audio del micrófono del navegador e integra
# las funciones should_end_turn() y should_interrupt() definidas arriba.
# Escucha en ws://0.0.0.0:8766/ws/mic

import asyncio
import os
import tempfile
import threading
import time

import numpy as np
import torchaudio
from fastapi import FastAPI, WebSocket
from fastapi.websockets import WebSocketDisconnect
import uvicorn

app_mic = FastAPI()

SAMPLING_RATE_MIC  = 16000
WINDOW_SAMPLES_MIC = 512   # ventana VAD a 16 kHz


@app_mic.websocket("/ws/mic")
async def websocket_mic_endpoint(websocket: WebSocket):
    """
    Endpoint WebSocket para audio del micrófono en continuo.

    - Recibe: chunks PCM int16 desde el navegador (cualquier tamaño).
    - Envía:  bytes WAV con la respuesta TTS  |  texto "INTERRUPT" como señal de control.
    """
    await websocket.accept()
    print("[MIC] Cliente conectado")

    vad_iterator     = VADIterator(VAD, sampling_rate=SAMPLING_RATE_MIC)
    speech_buffer    = []
    is_speaking      = False
    is_tts_playing   = False
    current_tts_text = ""
    speech_start_t   = None
    last_speech_t    = None

    # Contadores para detección de interrupción
    interrupt_frames    = 0
    interrupt_prob_sum  = 0.0

    loop = asyncio.get_event_loop()

    try:
        while True:
            raw = await websocket.receive_bytes()

            # Convertir PCM int16 → tensor float32 normalizado [-1, 1]
            audio_int16   = np.frombuffer(raw, dtype=np.int16)
            audio_float32 = audio_int16.astype(np.float32) / 32768.0
            chunk_tensor  = torch.from_numpy(audio_float32)
            now           = time.monotonic()

            for offset in range(0, len(chunk_tensor), WINDOW_SAMPLES_MIC):
                window = chunk_tensor[offset : offset + WINDOW_SAMPLES_MIC]
                if len(window) < WINDOW_SAMPLES_MIC:
                    break

                speech_prob  = VAD(window, SAMPLING_RATE_MIC).item()
                speech_event = vad_iterator(window, return_seconds=True)

                # ── Modo: TTS en reproducción → vigilar interrupción ──────────
                if is_tts_playing:
                    if speech_event and "start" in speech_event:
                        interrupt_frames   = 1
                        interrupt_prob_sum = speech_prob
                    elif interrupt_frames > 0:
                        interrupt_frames   += 1
                        interrupt_prob_sum += speech_prob
                        avg_prob = interrupt_prob_sum / interrupt_frames
                        if should_interrupt(interrupt_frames, avg_prob, current_tts_text):
                            is_tts_playing     = False
                            current_tts_text   = ""
                            interrupt_frames   = 0
                            interrupt_prob_sum = 0.0
                            await on_interruption_triggered(websocket, loop)
                            vad_iterator.reset_states()
                            speech_buffer = []
                            is_speaking   = False
                    continue  # no acumular buffer mientras TTS activo

                # ── Modo: escucha normal ──────────────────────────────────────
                if speech_event:
                    if "start" in speech_event:
                        is_speaking    = True
                        speech_start_t = now
                        last_speech_t  = now
                        speech_buffer  = [window]
                    elif "end" in speech_event and is_speaking:
                        last_speech_t = now   # inicio del silencio posterior
                elif is_speaking:
                    speech_buffer.append(window)
                    last_speech_t = now

                # ── Comprobar fin de turno ────────────────────────────────────
                if is_speaking and last_speech_t is not None:
                    silence_s = now - last_speech_t
                    speech_s  = (last_speech_t - speech_start_t) if speech_start_t else 0.0

                    if should_end_turn(silence_s, speech_s):
                        is_speaking = False
                        if speech_buffer:
                            full_audio    = torch.cat(speech_buffer)
                            speech_buffer = []

                            is_tts_playing   = True
                            interrupt_frames = 0
                            response_bytes, tts_text = await loop.run_in_executor(
                                None, _process_segment_with_text, full_audio
                            )
                            current_tts_text = tts_text or ""
                            if response_bytes:
                                await websocket.send_bytes(response_bytes)
                            else:
                                is_tts_playing = False

                        vad_iterator.reset_states()

    except WebSocketDisconnect:
        print("[MIC] Cliente desconectado")
    finally:
        vad_iterator.reset_states()


def _process_segment_with_text(audio_tensor: torch.Tensor):
    """
    Ejecuta ASR → LLM → TTS sobre un segmento de habla.
    Devuelve (bytes_wav, texto_tts) o (None, "") si no hay transcripción.
    """
    try:
        tmp_in = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp_in.close()
        torchaudio.save(tmp_in.name, audio_tensor.unsqueeze(0), SAMPLING_RATE_MIC)

        asr_result = run_asr(tmp_in.name)
        os.unlink(tmp_in.name)

        transcript = asr_result.get("text", "").strip()
        if not transcript:
            print("[ASR] Segmento sin texto, se descarta")
            return None, ""
        print(f"[ASR] {transcript}")

        llm_response = run_llm(transcript)
        print(f"[LLM] {llm_response}")

        tmp_out = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp_out.close()
        run_tts(llm_response, tmp_out.name)

        with open(tmp_out.name, "rb") as fh:
            wav_bytes = fh.read()
        os.unlink(tmp_out.name)

        return wav_bytes, llm_response

    except Exception as exc:
        print(f"[ERROR] Fallo al procesar segmento: {exc}")
        return None, ""


def _start_mic_server(host: str = "0.0.0.0", port: int = 8766):
    """Arranca el servidor FastAPI/uvicorn en un hilo daemon."""
    uvicorn.run(app_mic, host=host, port=port, log_level="info")


mic_server_thread = threading.Thread(target=_start_mic_server, daemon=True)
mic_server_thread.start()
print("✅ Servidor de micrófono arrancado en ws://0.0.0.0:8766/ws/mic")


In [ ]:
# --------------- CAPTURA DE MICRÓFONO DESDE EL NAVEGADOR (COLAB) ---------------
# Ejecuta esta celda para mostrar los controles de micrófono en el navegador.
# Pulsa "🎤 Iniciar", concede permisos de micrófono y empieza a hablar.
# El agente responderá en audio directamente en el navegador.
# Pulsa "⏹ Detener" para finalizar la sesión.

from IPython.display import display, Javascript

display(Javascript("""
(async () => {
  const WS_URL       = 'ws://localhost:8766/ws/mic';
  const SAMPLE_RATE  = 16000;
  const CHUNK_FRAMES = 512 * 20;  // ~640 ms por envío

  // ── Interfaz mínima ───────────────────────────────────────────────────────
  const box      = document.createElement('div');
  box.style.cssText = 'font-family:monospace;padding:10px;background:#f0f0f0;border-radius:6px;max-width:420px';
  const btnStart = document.createElement('button');
  const btnStop  = document.createElement('button');
  const status   = document.createElement('p');
  btnStart.textContent = '🎤 Iniciar';
  btnStop.textContent  = '⏹ Detener';
  btnStop.disabled     = true;
  status.textContent   = 'Estado: esperando...';
  box.appendChild(btnStart);
  box.appendChild(document.createTextNode(' '));
  box.appendChild(btnStop);
  box.appendChild(status);
  document.body.appendChild(box);

  let ws, audioCtx, source, processor, currentAudio = null;

  // ── Reproducir respuesta WAV ──────────────────────────────────────────────
  async function playWav(arrayBuffer) {
    if (currentAudio) { currentAudio.stop(); currentAudio = null; }
    try {
      const decoded = await audioCtx.decodeAudioData(arrayBuffer);
      const src     = audioCtx.createBufferSource();
      src.buffer    = decoded;
      src.connect(audioCtx.destination);
      src.start();
      currentAudio  = src;
      src.onended   = () => { currentAudio = null; status.textContent = 'Estado: 🔴 escuchando'; };
    } catch (e) {
      console.error('Error al reproducir WAV:', e);
    }
  }

  // ── Iniciar captura ───────────────────────────────────────────────────────
  btnStart.onclick = async () => {
    btnStart.disabled = true;
    btnStop.disabled  = false;
    status.textContent = 'Estado: conectando...';

    ws = new WebSocket(WS_URL);
    ws.binaryType = 'arraybuffer';

    ws.onopen  = () => { status.textContent = 'Estado: 🔴 escuchando'; };
    ws.onclose = () => { status.textContent = 'Estado: conexión cerrada'; btnStart.disabled = false; btnStop.disabled = true; };
    ws.onerror = (e) => { status.textContent = 'Estado: ❌ error de conexión'; console.error(e); };

    ws.onmessage = async (evt) => {
      if (typeof evt.data === 'string' && evt.data === 'INTERRUPT') {
        // Señal de interrupción: detener reproducción actual
        if (currentAudio) { currentAudio.stop(); currentAudio = null; }
        status.textContent = 'Estado: 🔴 escuchando (interrumpido)';
      } else if (evt.data instanceof ArrayBuffer) {
        status.textContent = '🔊 reproduciendo respuesta...';
        await playWav(evt.data);
      }
    };

    // Inicializar contexto de audio
    audioCtx  = new (window.AudioContext || window.webkitAudioContext)({ sampleRate: SAMPLE_RATE });
    const stream = await navigator.mediaDevices.getUserMedia({
      audio: { sampleRate: SAMPLE_RATE, channelCount: 1, echoCancellation: true, noiseSuppression: true }
    });

    source    = audioCtx.createMediaStreamSource(stream);
    processor = audioCtx.createScriptProcessor(CHUNK_FRAMES, 1, 1);

    processor.onaudioprocess = (e) => {
      if (!ws || ws.readyState !== WebSocket.OPEN) return;
      const float32 = e.inputBuffer.getChannelData(0);
      const int16   = new Int16Array(float32.length);
      for (let i = 0; i < float32.length; i++) {
        int16[i] = Math.max(-32768, Math.min(32767, Math.round(float32[i] * 32768)));
      }
      ws.send(int16.buffer);
    };

    source.connect(processor);
    processor.connect(audioCtx.destination);
  };

  // ── Detener captura ───────────────────────────────────────────────────────
  btnStop.onclick = () => {
    if (processor)    { processor.onaudioprocess = null; processor.disconnect(); }
    if (source)       { source.disconnect(); }
    if (currentAudio) { currentAudio.stop(); currentAudio = null; }
    if (audioCtx)     { audioCtx.close(); }
    if (ws)           { ws.close(); }
    btnStart.disabled = false;
    btnStop.disabled  = true;
    status.textContent = 'Estado: detenido';
  };
})();
"""))
